# 9.2 Random Forests

Im letzten Kapitel haben wir fünf Ensemble-Methoden in der Theorie
kennengelernt: Voting, Averaging, Stacking, Bagging und Boosting. Für Bagging und
Boosting werden besonders häufig Entscheidungsbäume eingesetzt. In diesem Kapitel
betrachten wir die praktische Umsetzung von Bagging mit Entscheidungsbäumen, die
sogenannten **Random Forests**.

## Random Forests mit Scikit-Learn

Entscheidungsbäume (Decision Trees) haben wir bereits betrachtet. Sie sind
aufgrund ihrer Einfachheit und vor allem aufgrund ihrer Interpretierbarkeit sehr
beliebt. Allerdings ist ihre Tendenz zum Overfitting problematisch. Daher
kombinieren wir die Ensemble-Methode Bagging mit Entscheidungsbäumen. Indem aus
den Trainingsdaten zufällig Bootstrap-Stichproben ausgewählt werden, erhalten
wir unterschiedliche Entscheidungsbäume. Zusätzlich wird beim Training nicht mit
allen Merkmalen (Features) trainiert. Bei jedem Split eines Baumes wird nur eine
zufällige Teilmenge der Merkmale betrachtet, um die beste Trennung zu finden.
Durch diese zwei Maßnahmen wird die Anpassung der Entscheidungsbäume an die
Trainingsdaten (Overfitting) reduziert.

Um den Random Forest von Scikit-Learn praktisch auszuprobieren, erzeugen wir
künstliche Daten. Dazu verwenden wir die Funktion `make_moons` von Scikit-Learn.
Sie erzeugt zwei ineinander verschlungene Punktwolken, die zu zwei Klassen
gehören. Die beiden Koordinaten der Punkte interpretieren wir als Kilometerstand
und Preis von Autos bei einer fiktiven Verkaufsaktion. Die zugehörige Klasse
interpretieren wir als "verkauft" (1) oder "nicht verkauft" (0).

In [ ]:
import pandas as pd 
from sklearn.datasets import make_moons

# Erzeugung künstlicher Daten
X_array, y_array = make_moons(n_samples=120, random_state=0, noise=0.3)

daten = pd.DataFrame({
    'Kilometerstand [km]': 10000 * (X_array[:,0] + 2),
    'Preis [EUR]': 5000 * (X_array[:,1] + 2),
    'verkauft': y_array,
    })

# Daten ins richtige Format bringen
X = daten[['Kilometerstand [km]', 'Preis [EUR]']].values
y = daten['verkauft'].values

Diesmal werden in dem Autohaus 120 Autos zum Verkauf angeboten (siehe Option
`n_samples=120`). Nach Aktionsende werden die Merkmale Kilometerstand und Preis
tabellarisch erfasst und notiert, ob das Auto verkauft wurde (1) oder nicht
verkauft wurde (0). Wir visualisieren die Daten.

In [ ]:
import plotly.express as px
# plot artificial data
fig = px.scatter(daten, x = 'Kilometerstand [km]', y = 'Preis [EUR]', color=daten['verkauft'].astype(bool),
        title='Künstliche Daten: Verkaufsaktion Autohaus',
        labels = {'x': 'Kilometerstand [km]', 'y': 'Preis [EUR]', 'color': 'verkauft'})
fig.show()

Nachdem wir die Vorbereitungen für die Daten abgeschlossen haben, können wir
Scikit-Learn einen Random Forest trainieren lassen. Dazu importieren wir den
Algorithmus aus dem Modul `ensemble`. Da der Random Forest ein Ensemble von
Entscheidungsbäumen ist, haben wir nun die Möglichkeit, die Anzahl der
Entscheidungsbäume festzulegen. Voreingestellt sind 100
Entscheidungsbäume. Aus didaktischen Gründen reduzieren wir diese Anzahl auf
vier und setzen das Argument `n_estimators=` auf `4`. Zusätzlich setzen wir
`random_state=0`. Damit liegen die Zufallszahlen für das Bootstrapping und die
Merkmalsauswahl fest, und bei allen entsteht dasselbe Ergebnis. Auch in echten
Projekten fixieren wir diesen Wert, damit die Ergebnisse reproduzierbar bleiben.
Wir verlassen uns aber nicht darauf, dass ausgerechnet dieser Wert das beste
Modell liefert. Zuletzt führen wir das Training mit der `.fit()`-Methode durch.
Weitere Details finden Sie unter [Scikit-Learn Dokumentation →
RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

wald = # TODO: ???  (Random Forest mit 4 Bäumen anlegen, reproduzierbar mit dem Wert 0)
wald.fit(X, y)

Als Nächstes lassen wir den Random Forest für jeden Punkt des Gebiets
prognostizieren, ob ein Auto mit diesem Kilometerstand und diesem Preis
verkaufbar wäre oder nicht. Dafür verwenden wir die Funktion
`DecisionBoundaryDisplay` aus Scikit-Learn. Diese Funktion baut auf Matplotlib
auf, nicht auf Plotly, und der Code sieht deshalb etwas anders aus als sonst im
Skript. Sie müssen diesen Code nicht selbst schreiben können. Wichtig ist nur,
dass Sie das Bild lesen können. Die eingefärbten Flächen zeigen, wie der Random
Forest jeden Punkt aus Kilometerstand und Preis einordnet.

In [ ]:
from matplotlib.colors import ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay

flaechen_farben = ListedColormap(['#EF553B33', '#636EFA33'])
punkt_farben = ListedColormap(['#EF553B', '#636EFA'])

fig = DecisionBoundaryDisplay.from_estimator(wald, X,
    cmap=flaechen_farben, grid_resolution=1000)
fig.ax_.scatter(X[:,0], X[:,1], c=y, cmap=punkt_farben)
fig.ax_.set_xlabel('Kilometerstand [km]');
fig.ax_.set_ylabel('Preis [EUR]');
fig.ax_.set_title('Random Forest: Entscheidungsgrenzen');

Auf die vier Entscheidungsbäume, aus denen der Random Forest kombiniert wurde,
greifen wir mit dem Attribut `estimators_` zu. Wir lassen uns jetzt die
Entscheidungsgrenzen einzeln für jeden Entscheidungsbaum anzeigen.

In [ ]:
flaechen_farben = ListedColormap(['#EF553B33', '#636EFA33'])
punkt_farben = ListedColormap(['#EF553B', '#636EFA'])

nummer = 1
for baum in wald.estimators_:
    fig = DecisionBoundaryDisplay.from_estimator(baum, X,
        cmap=flaechen_farben, grid_resolution=1000)
    fig.ax_.scatter(X[:,0], X[:,1], c=y, cmap=punkt_farben)
    fig.ax_.set_xlabel('Kilometerstand [km]');
    fig.ax_.set_ylabel('Preis [EUR]');
    fig.ax_.set_title(f'Entscheidungsbaum {nummer}');
    nummer = nummer + 1

## Weniger Overfitting als ein einzelner Baum

In Kapitel 6 haben wir gesehen, dass ein Entscheidungsbaum die Trainingsdaten
sehr genau nachbildet und dabei zu Overfitting neigt. Wir prüfen jetzt mit einem
Train-Test-Split, ob der Random Forest besser mit neuen Daten zurechtkommt. Dazu
teilen wir die Daten mit `train_test_split()` in Trainings- und Testdaten auf,
trainieren einen einzelnen Entscheidungsbaum und einen Random Forest nur auf den
Trainingsdaten und vergleichen den Score auf beiden Teilen.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = # TODO: ???  (Daten aufteilen mit 20 % Testdaten, reproduzierbar mit dem Wert 0)

baum = DecisionTreeClassifier(random_state=0)
baum.fit(X_train, y_train)
print(f'Baum - Training: {baum.score(X_train, y_train):.2f}, Test: {baum.score(X_test, y_test):.2f}')

wald = RandomForestClassifier(n_estimators=4, random_state=0)
# TODO: ???  (Random Forest nur auf den Trainingsdaten trainieren)
print(f'Wald - Training: {wald.score(X_train, y_train):.2f}, Test: {wald.score(X_test, y_test):.2f}')

Der einzelne Entscheidungsbaum erreicht auf den Trainingsdaten einen perfekten
Score von 1.00, auf den Testdaten aber nur etwa 0.75. Diese große Lücke zwischen
Training und Test ist das typische Zeichen für Overfitting. Der Random Forest
erreicht auf den Trainingsdaten einen etwas niedrigeren Score, auf den Testdaten
aber einen höheren Wert von etwa 0.83. Training und Test liegen näher
beieinander. Das Modell überträgt sich also besser auf neue Daten.

Der Grund liegt in den beiden Zufallsmechanismen des Random Forests. Jeder Baum
sieht nur eine Bootstrap-Stichprobe der Trainingsdaten und bei jedem Split nur
eine zufällige Teilmenge der Merkmale. Die einzelnen Bäume passen sich weiterhin
zu stark an ihre jeweiligen Daten an, aber jeder auf seine eigene Weise. Wenn
wir die Prognosen mitteln, heben sich diese zufälligen Abweichungen teilweise
auf. Aus vielen leicht überangepassten Bäumen wird so ein stabileres
Gesamtmodell.

## Feature Importance

Wie wir gerade gesehen haben, reduziert der Random Forest das Overfitting und
ist damit für zukünftige Prognosen besser gerüstet. Er verliert aber seine
leichte Interpretierbarkeit. Ein großer Vorteil des Entscheidungsbaumes ist,
dass wir die Entscheidungen als eine Abfolge von Entscheidungsfragen gut
nachvollziehen können. Jeder der vielen Bäume im Random Forest kommt jedoch zu
einer anderen Reihenfolge der Entscheidungsfragen und zu anderen Grenzen. Dafür
bietet der Random Forest eine alternative Bewertung, wie wichtig einzelne
Merkmale (Features) sind, die sogenannte **Feature Importance**.

Die Feature Importance bewertet, wie stark ein Merkmal zur Prognosequalität
beiträgt. Ein höherer Wert bedeutet mehr Einfluss. Bei einem
Entscheidungsbaum wird dazu für jedes Merkmal aufsummiert, wie stark die Splits
mit diesem Merkmal die Gini-Impurity reduzieren. Bei einem einzelnen Baum ist
dieser Wert wenig verlässlich, denn er hängt stark von der zufälligen Stichprobe
und der zufälligen Merkmalsauswahl ab. Ein Random Forest trainiert viele Bäume.
Wenn wir die Feature Importance über alle Bäume mitteln, erhalten wir ein
deutlich stabileres Bewertungskriterium.

Wir trainieren nun einen Random Forest mit der Standardeinstellung von 100
Entscheidungsbäumen und lassen uns dann die Feature Importance ausgeben. Nach
dem Training steht sie im Attribut `feature_importances_` des Modells.

In [ ]:
wald = RandomForestClassifier(random_state=0)
wald.fit(X, y)

# TODO: ???  (Feature Importance des Random Forests ausgeben)

Der erste Wert gibt die Feature Importance für das erste Merkmal an und der
zweite Wert entsprechend für das zweite Merkmal. Es ist üblich, die Feature
Importance als Balkendiagramm zu visualisieren.

In [ ]:
feature_importances = pd.Series(wald.feature_importances_, index=['Kilometerstand [km]', 'Preis [EUR]'])

fig = px.bar(feature_importances, orientation='h',
  title='Verkaufsaktion im Autohaus', 
  labels={'value':'Feature Importance', 'index': 'Merkmal'})
fig.update_traces(showlegend=False) 
fig.show()

Der Preis trägt hier also etwas mehr zur Prognose bei als der Kilometerstand.
Wir sollten diese Werte aber nicht überbewerten. Die Berechnung über die
Gini-Reduktion bevorzugt Merkmale mit vielen verschiedenen Werten, zum Beispiel
fein aufgelöste Messgrößen. Die Feature Importance gibt daher eher eine Tendenz
an als eine exakte Rangfolge.

## Zusammenfassung und Ausblick

Random Forests sind einfachen Entscheidungsbäumen meist vorzuziehen, da sie das
Overfitting reduzieren. Die einzelnen Entscheidungsbäume können parallel
trainiert werden, so dass das Training auch bei vielen Bäumen schnell geht. Über
die Feature Importance liefert der Random Forest zusätzlich eine grobe
Einordnung, welche Merkmale den größten Einfluss haben.

Random Forests setzen die Ensemble-Methode Bagging um. Im nächsten Kapitel
betrachten wir mit **XGBoost** einen Vertreter der Boosting-Methoden. Dort werden
die Entscheidungsbäume nicht parallel trainiert und gemittelt, sondern
nacheinander, wobei jeder neue Baum die Fehler seiner Vorgänger korrigiert.

## Mini-Übungen

### Mini-Übung 1

Der oben trainierte Random Forest besteht aus vier Entscheidungsbäumen.

1. Wie viele Einzelprognosen liefern diese Bäume für ein einzelnes neues Auto,
   und wie entsteht daraus die finale Prognose des Random Forests?
2. Sehen Sie sich die Entscheidungsgrenzen der vier einzelnen Bäume an. Warum
   unterscheiden sie sich, obwohl derselbe Algorithmus und dieselben
   Trainingsdaten verwendet wurden?
3. Trainieren Sie den Random Forest mehrmals ohne die Einstellung
   `random_state=0`. Was beobachten Sie?

### Mini-Übung 2

Vergleichen Sie einen einzelnen Entscheidungsbaum und einen Random Forest auf
denselben Trainings- und Testdaten.

1. Bei welchem der beiden Modelle liegen Trainings-Score und Test-Score weiter
   auseinander? Was bedeutet das?
2. Erhöhen Sie die Anzahl der Bäume im Random Forest von 4 auf 100. Wird der
   Test-Score dadurch immer weiter besser?
3. Erklären Sie in eigenen Worten, warum das Mitteln vieler Bäume das
   Overfitting verringert.

### Mini-Übung 3

1. Welchen Wert ergeben die Feature Importances von Preis und Kilometerstand
   zusammen?
2. In einem echten Projekt schwanken die Feature-Importance-Werte eines
   einzelnen Entscheidungsbaums bei jedem neuen Training stark. Wie hilft ein
   Random Forest hier weiter?
3. Ein Merkmal hat die Feature Importance 0. Bedeutet das sicher, dass es für
   die Zielgröße unwichtig ist?